# L7.2 · SmolVLA: la imagen, de foto a 64 tokens

| Notebook | Qué abre |
|---|---|
| L7.1 | El mapa: qué entra, qué sale, dónde están los 450 M de parámetros |
| **L7.2** | La imagen: de la foto a 64 tokens |
| L7.3 | La fila del VLM: palabras, estado y quién puede mirar a quién |
| L7.4 | Adentro del VLM: qué miran las palabras, por qué 16 de 32 capas, qué queda guardado |
| `06` (flujo) | Entrenarlo con nuestros datos |
| L7.5 | El experto de acciones: la brújula de L6 con ojos |
| L7.6 | Las 10 pasadas: un bloque de verdad pasando de ruido a movimiento |
| L7.7 | El entrenamiento por dentro |
| `07` (flujo) | Evaluación contra ACT, en los mismos episodios |

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo

import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import DATASETS_DIR
from toolkit.policy import smolvla_internals as vla
from toolkit.policy.smolvla_g1_base import G1_BASE_DIR, load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"

ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
CAMERA = "observation.images.cam_right_high"
dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT)
# El VLM queda congelado al entrenar: el modelo base y el entrenado ven igual.
policy, preprocess, _ = load_g1_policy(G1_BASE_DIR, dataset.meta.stats, device)
model = policy.model

frame = dataset[int(dataset.meta.episodes["dataset_from_index"][10]) + 120]  # el instante de L7.1
obs = vla.prepare_observation(policy, preprocess(dict(frame)))
image = obs.images[0]  # (1, 3, 512, 512), ya rellenada y en [-1, 1]


def show(ax, img, title=None):
    """Una imagen en [-1, 1] como foto."""
    ax.imshow(((img[0].permute(1, 2, 0).float().cpu() + 1) / 2).clamp(0, 1).numpy())
    ax.axis("off")
    if title:
        ax.set_title(title)

## 1. El problema

El VLM es un transformer: entiende **filas de vectores**, y nada más. Una foto
de la cámara son 640 × 360 píxeles × 3 colores = **691.200 números**. Meterlos
uno por uno sería una fila imposible de procesar, y cada píxel solo no dice
nada: un píxel azul no es un mameluco.

Lo que hace falta es pocos vectores, y que cada uno diga **qué hay en una zona
de la foto**. SmolVLA lo resuelve en cinco pasos, y termina con 64 tokens.

## 2. Paso 1: llevarla a 512 × 512 sin deformarla

El codificador de imágenes (SigLIP) se entrenó con fotos cuadradas de 512 ×
512. Estirar la nuestra deformaría las formas, así que se achica hasta que
entre y se **rellena** lo que falta con negro, arriba y a la izquierda. Además,
los colores pasan de [0, 1] a [-1, 1], que es como los espera SigLIP.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(frame[CAMERA].permute(1, 2, 0).numpy())
axes[0].set_title(f"original: {frame[CAMERA].shape[2]} × {frame[CAMERA].shape[1]}")
axes[0].axis("off")
show(axes[1], image, "lo que entra a SigLIP: 512 × 512")
padding_rows = int((image[0].float() <= -0.999).all(dim=(0, 2)).sum())
print(f"filas de relleno arriba: {padding_rows} de 512 → {padding_rows / 512:.0%} del cuadrado es negro")

## 3. Paso 2: cortarla en parches

SigLIP corta el cuadrado en parches de 16 × 16 píxeles: una grilla de **32 ×
32 = 1024 parches**. Cada parche será un vector.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
show(ax, image, "1024 parches de 16 × 16 píxeles")
for k in range(0, 513, 16):
    ax.axhline(k - 0.5, color="white", lw=0.3)
    ax.axvline(k - 0.5, color="white", lw=0.3)
padding_patch_rows = padding_rows // 16
print(f"{padding_patch_rows} filas enteras de parches son relleno: {padding_patch_rows * 32} de 1024 parches")

## 4. Paso 3: SigLIP describe cada parche

SigLIP es un transformer de 12 capas que se entrenó con millones de pares
(imagen, texto). Los 1024 parches entran como fila, en cada capa se miran entre
sí, y al final **cada parche sale descripto con 768 números** que ya no dicen
"qué color tiene" sino "qué hay acá", con el contexto del resto de la foto.

¿Cómo se ve algo de 768 dimensiones? Un truco clásico: buscar las 3
direcciones en las que más varían los parches (PCA) y pintarlas como rojo,
verde y azul. Parches que SigLIP describe parecido quedan del mismo color.

In [ ]:
with torch.no_grad():
    patches = vla.vision_patches(model, image)[0].float().cpu()  # (1024, 768)
print("salida de SigLIP:", tuple(patches.shape))

is_padding = torch.zeros(32, 32, dtype=torch.bool)
is_padding[:padding_patch_rows] = True
is_padding = is_padding.flatten()
# Todos los vectores de SigLIP comparten un componente grande y común: sin restarlo, todo se parece a todo.
# Restamos el promedio de los parches con foto y los llevamos a largo 1, para comparar sólo direcciones.
mean = patches[~is_padding].mean(0)
features = torch.nn.functional.normalize(patches - mean, dim=1)
_, _, directions = torch.linalg.svd(features[~is_padding], full_matrices=False)
projected = features @ directions[:3].T
low, high = projected[~is_padding].quantile(0.02, dim=0), projected[~is_padding].quantile(0.98, dim=0)
colors = ((projected - low) / (high - low)).clamp(0, 1)
colors[is_padding] = 0.5

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
show(axes[0], image, "la foto")
axes[1].imshow(colors.reshape(32, 32, 3).numpy(), interpolation="nearest")
axes[1].set_title("cómo describe SigLIP cada parche (3 direcciones principales)")
axes[1].axis("off")

Sin que nadie le diga qué es cada cosa, SigLIP separa zonas: las paredes del
estante a los costados (azul y violeta), el paquete en el centro (celeste y
verde claro), y abajo, en los costados, los brazos y las manos (verde oscuro).
El relleno está en gris porque lo dejamos afuera de la cuenta.

Otra forma de mirar lo mismo: elegimos un parche y medimos **cuánto se parece
su descripción** a la de todos los demás (similitud coseno, con el promedio
restado: 1 = misma dirección, 0 = nada que ver). El relleno no se pinta.

In [ ]:
def similarity_map(row, col):
    # Promedio de un bloque de 3 × 3 parches alrededor del elegido: un parche solo es ruidoso.
    neighbors = [r * 32 + c for r in range(row - 1, row + 2) for c in range(col - 1, col + 2)]
    probe = torch.nn.functional.normalize(features[neighbors].mean(0), dim=0)
    similarity = features @ probe  # con el promedio restado (sección 4)
    similarity[is_padding] = float("nan")  # el relleno no se pinta
    return similarity.reshape(32, 32)


probes = {"el mameluco": (24, 16), "la mano derecha": (28, 28), "la pared del estante": (18, 29)}
fig, axes = plt.subplots(1, len(probes), figsize=(16, 5.2))
for ax, (name, (row, col)) in zip(axes, probes.items()):
    show(ax, image)
    ax.imshow(similarity_map(row, col).numpy(), cmap="magma", alpha=0.7, extent=(0, 512, 512, 0), vmin=-0.2, vmax=0.8)
    ax.add_patch(plt.Rectangle(((col - 1) * 16, (row - 1) * 16), 48, 48, fill=False, edgecolor="cyan", lw=2))
    ax.set_title(f"parecido con un parche de {name}")
plt.tight_layout()

Se lee, aunque con ruido: el parche del mameluco se parece sobre todo al
resto del paquete, y el de la pared a las paredes y al borde del estante. Las
dos manos se parecen entre sí menos de lo que uno esperaría. Es razonable:
SigLIP se entrenó para emparejar **la imagen entera** con un texto, no para
describir cada zona con precisión, y sus parches son más ruidosos que los de
modelos entrenados para eso. El paper de SmolVLA lo nombra como limitación: su
VLM se preentrenó sobre todo con documentos y lectura de texto, no con escenas
de robots.

## 5. Paso 4: juntar de a 16 (pixel shuffle)

1024 tokens por imagen son muchos para la fila del VLM. El **conector** los
reduce juntando cada bloque de **4 × 4 parches vecinos** en un solo vector:
pega los 16 vectores de 768 uno detrás del otro, y queda uno de 16 × 768 =
**12.288** números. La grilla pasa de 32 × 32 a **8 × 8 = 64 tokens**.

No se pierde información: no se promedia nada, sólo se reacomoda. Lo
comprobamos: el primer pedazo del token 0 es exactamente el parche (0, 0), y
el segundo pedazo es el parche (0, 1).

In [ ]:
with torch.no_grad():
    shuffled = vla.pixel_shuffle(model, vla.vision_patches(model, image))[0].float().cpu()  # (64, 12288)
print("después de juntar:", tuple(shuffled.shape))
print("primer pedazo del token 0 == parche (0, 0):", torch.equal(shuffled[0, :768], patches[0]))
print("segundo pedazo del token 0 == parche (0, 1):", torch.equal(shuffled[0, 768:1536], patches[1]))

fig, ax = plt.subplots(figsize=(6, 6))
show(ax, image, "64 tokens: cada uno junta 4 × 4 parches")
for k in range(0, 513, 64):
    ax.axhline(k - 0.5, color="yellow", lw=1.2)
    ax.axvline(k - 0.5, color="yellow", lw=1.2)
for k in range(0, 513, 16):
    ax.axhline(k - 0.5, color="white", lw=0.2)
    ax.axvline(k - 0.5, color="white", lw=0.2)
token_rows_padding = padding_rows // 64
print(f"tokens que ven sólo relleno: {token_rows_padding * 8} de 64; "
      f"la fila {token_rows_padding} mezcla relleno ({padding_rows % 64} píxeles) y foto")

De los 64 tokens, 24 ven sólo relleno negro y otra fila de 8 mezcla relleno
con foto: más de un tercio de la parte visual de la fila se gasta en nada. El
modelo igual los procesa; no hay máscara que los saque.

## 6. Paso 5: traducir al idioma del VLM de texto

Cada token tiene ahora 12.288 números, pero el VLM de texto trabaja con
vectores de **960**. Una sola matriz de 12.288 × 960 (los 11,8 M de
parámetros del conector) hace la traducción. Después se multiplica por √960,
el mismo escalado que reciben los vectores de las palabras. ¿Quedan del mismo
tamaño que las palabras?

In [ ]:
with torch.no_grad():
    tokens = vla.image_tokens(model, image)[0].float().cpu()  # (64, 960)
    words = (model.vlm_with_expert.embed_language_tokens(obs.lang_tokens) * 960 ** 0.5)[0].float().cpu()
real = obs.lang_masks[0].cpu()
print("tokens de imagen:", tuple(tokens.shape), "· matriz del conector:",
      tuple(model.vlm_with_expert.get_vlm_model().connector.modality_projection.proj.weight.T.shape))

image_lengths, word_lengths = tokens.norm(dim=1), words[real].norm(dim=1)
fig, ax = plt.subplots(figsize=(8, 3))
bins = np.logspace(0, 4.5, 60)
ax.hist(image_lengths.numpy(), bins=bins, alpha=0.7, label="64 tokens de imagen")
ax.hist(word_lengths.numpy(), bins=bins, alpha=0.7, label=f"{int(real.sum())} tokens de palabras")
ax.set_xscale("log")
ax.set_xlabel("largo del vector (escala logarítmica)")
ax.legend()
plt.tight_layout()
print(f"largo típico · imagen {image_lengths.median():.0f} · palabras {word_lengths.median():.0f} "
      f"→ la imagen entra {image_lengths.median() / word_lengths.median():.0f} veces más larga")

No: los tokens de imagen entran unas **40 veces más largos** que los de
palabras. No es un error de SmolVLA: el escalado multiplica a los dos por lo
mismo, así que la proporción es la misma con la que se entrenó SmolVLM, que
mete los tokens de imagen entre las palabras sin escalar ninguno (lo
verificamos en su código, `inputs_merger`). Y la atención no se rompe: cada
capa empieza normalizando cada token a un largo parecido (una normalización
llamada RMSNorm) antes de compararlo con los demás.

## Lo que nos llevamos

- La foto se rellena hasta 512 × 512, se corta en 1024 parches, SigLIP
  describe cada uno con 768 números, el conector los junta de a 16 y los
  traduce a 64 tokens de 960.
- Los tokens describen **qué hay** en cada zona, no colores: zonas parecidas
  (las dos manos, la tela) quedan con descripciones parecidas.
- En nuestro caso, un tercio de los tokens describe el relleno negro. Es una
  consecuencia de la cámara apaisada, no un error; recortar en vez de rellenar
  sería un experimento posible, de una sola variable.
- ACT, para comparar, sacaba 240 tokens de la misma foto con una ResNet de
  ImageNet (L3.3): más tokens, pero de un modelo que nunca vio texto.

## Referencias

- Zhai et al., [SigLIP](https://arxiv.org/abs/2303.15343) (2023): el
  codificador de imágenes.
- Marafioti et al., [SmolVLM](https://arxiv.org/abs/2504.05299) (2025): el
  VLM y su pixel shuffle.
- Shukor et al., [SmolVLA](https://arxiv.org/abs/2506.01844) (2025), sección
  3.1: sin mosaicos, sólo la imagen global y 64 tokens por cuadro.